# CozyDay P1 A8 — Stage 1
**Owner: Lin Gao** | Step 1.1 Knowledge Base and Step 1.2 Chunking only. Stages 2–4 will be added by teammates after Stage 1 is merged.

## 1. Load and validate the knowledge base
30 original English paragraphs with stable IDs, titles, topics, and bodies. All filesystem paths are relative to this notebook / module directory.

In [1]:
from chunking import (load_knowledge_base, validate_knowledge_base,
                      fixed_length_chunks, overlapping_paragraph_chunks, hybrid_topic_chunks,
                      build_all_chunks, validate_chunks)
records = load_knowledge_base()
validation = validate_knowledge_base(records)
print(validation)
print("First paragraph:", records[0]["id"], records[0]["title"])

{'paragraphs': 30, 'total_words': 3111, 'min_words': 97, 'max_words': 112, 'unique_ids': 30}
First paragraph: P001 Creating an actionable task


## 2. Three deterministic chunking strategies
**Fixed:** whole-sentence packing to 130 words, flushing at fine-topic boundaries. **Overlap:** adjacent whole-paragraph pairs inside documented related sections, stride 1. **Hybrid:** nonoverlapping whole paragraphs with identical fine topics, soft cap 230 words.

All examples below show complete chunk text. See `rag_analysis.md` for the semantic section mapping and soft-cap exceptions.

In [2]:
chunks_by_strategy = build_all_chunks(records)
for strategy, chunks in chunks_by_strategy.items():
    print(f"{strategy}: {len(chunks)} chunks")
    example = chunks[0]
    print("ID:", example["chunk_id"], "| Topic:", example["topic"])
    print("Source IDs:", example["source_ids"], "| Words:", example["word_count"])
    print("Complete example:\n" + example["text"] + "\n")
print("Actual overlapping windows in Task Organization:")
for chunk in chunks_by_strategy["overlap"][:3]:
    print(chunk["chunk_id"], chunk["source_ids"], chunk["topic"])

fixed: 30 chunks
ID: FIXED_001 | Topic: Task Creation
Source IDs: ['P001'] | Words: 112
Complete example:
A useful task describes one clear outcome rather than a vague intention. For example, replace “work on biology” with “summarize chapter four and answer the practice questions.” When preparing a task in CozyDay, give it a short descriptive title, optional notes, an estimated duration, and a realistic due date if one exists. Record the next physical action in the notes when the work feels overwhelming. Large assignments should become several smaller tasks, each with an observable completion criterion. Review a newly entered task to make sure its wording still makes sense when seen without the surrounding conversation. An organized task list supports planning, but creating tasks does not replace doing the work.

overlap: 19 chunks
ID: OVERLAP_001 | Topic: Task Creation
Source IDs: ['P001', 'P002'] | Words: 216
Complete example:
A useful task describes one clear outcome rather than a v

## 3. Automated checks and Stage 2 interface
Validation checks text preservation, exact source provenance, metadata, semantic boundaries, and coverage. The standalone regression suite is run with `python test_stage1.py`. Stage 2 embeds `chunk["text"]` and preserves the other fields. Full-corpus size validation is separate from chunking so a small scaling subset remains supported.

In [3]:
for strategy, chunks in chunks_by_strategy.items():
    validate_chunks(records, chunks, strategy)
print("Stage 1 checks PASSED")
print("Chunk object keys:", list(chunks_by_strategy["fixed"][0]))
print("Five-paragraph scaling subset:",
      {name: len(chunks) for name, chunks in build_all_chunks(records[:5]).items()})

Stage 1 checks PASSED
Chunk object keys: ['chunk_id', 'strategy', 'source_ids', 'topic', 'text', 'word_count']
Five-paragraph scaling subset: {'fixed': 5, 'overlap': 4, 'hybrid': 3}


# CozyDay P1 A8 -- Stage 2
**Owner: Rishabh Puri** | Step 1.3 Embedding Pipeline, Step 1.4 Retrieval System, Step 1.5 Generation Pipeline, Step 2.1 Test Queries.

Builds directly on Stage 1's frozen interface above (`records`, `chunks_by_strategy`). The pipeline logic lives in five small modules next to this notebook -- `embedding.py`, `retrieval.py`, `generation.py`, `queries.py`, `rag.py` -- so Stage 3 and Stage 4 can import the same functions instead of re-deriving them.

## 4. Embedding pipeline (Step 1.3)
Three sentence-transformers models fixed by the team's shared contract -- `all-MiniLM-L6-v2` (384d), `all-mpnet-base-v2` (768d), `bge-large-en-v1.5` (1024d) -- loaded once, with device selection (cuda -> mps -> cpu) and each model's *actual* output dimension checked against the expected dimensionality. Every chunk from all three Stage 1 strategies is embedded for all three models (9 combinations total) and cached to `results/embeddings/` so re-running this cell does not recompute vectors that are already on disk.

In [4]:
from embedding import load_all_embedding_models, describe_models, embed_all_combinations

model_infos = load_all_embedding_models()
print("Embedding models loaded:")
for row in describe_models(model_infos):
    print(" ", row)

embeddings = embed_all_combinations(model_infos, chunks_by_strategy)
print("\nEmbedded all 3 models x 3 chunking strategies = 9 combinations:")
for model_key, by_strategy in embeddings.items():
    for strategy, result in by_strategy.items():
        status = "cached" if result["from_cache"] else f"{result['embed_time_seconds']:.3f}s"
        print(f"  {model_key:7s} {strategy:8s} {result['vectors'].shape}  {status}")


/Users/rishabhpuri/PycharmProjects/3_CozyDay/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1028 17:41:26.296000 36613 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|                                                                                                                                 | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 10036.08it/s]

Loading weights:   0%|                                                                                                                                 | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 8781.53it/s]

Loading weights:   0%|                                                                                                                                 | 0/391 [00:00<?, ?it/s]

Loading weights: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 391/391 [00:00<00:00, 10047.25it/s]

Embedding models loaded:
  {'key': 'small', 'model_name': 'sentence-transformers/all-MiniLM-L6-v2', 'dimension': 384, 'device': 'mps', 'load_time_seconds': 0.6769464169628918}
  {'key': 'medium', 'model_name': 'sentence-transformers/all-mpnet-base-v2', 'dimension': 768, 'device': 'mps', 'load_time_seconds': 0.6906309998594224}
  {'key': 'large', 'model_name': 'BAAI/bge-large-en-v1.5', 'dimension': 1024, 'device': 'mps', 'load_time_seconds': 1.5578322501387447}

Embedded all 3 models x 3 chunking strategies = 9 combinations:
  small   fixed    (30, 384)  cached
  small   overlap  (19, 384)  cached
  small   hybrid   (15, 384)  cached
  medium  fixed    (30, 768)  cached
  medium  overlap  (19, 768)  cached
  medium  hybrid   (15, 768)  cached
  large   fixed    (30, 1024)  cached
  large   overlap  (19, 1024)  cached
  large   hybrid   (15, 1024)  cached


## 5. Retrieval system (Step 1.4)
`retrieve(query, model_info, strategy, embeddings, chunk_lookup, top_k)` is the one consistent entry point used for every experiment in this assignment. It embeds the query with the requested model, ranks cached chunk vectors with scikit-learn's `cosine_similarity` (the searchable structure the assignment asks for), and returns chunk IDs, complete chunk text, similarity scores, and both the embedding- and search-latency separately.

In [5]:
from retrieval import build_chunk_lookup, retrieve
from queries import TEST_QUERIES, validate_test_queries

validate_test_queries(records)
chunk_lookup = build_chunk_lookup(chunks_by_strategy)
print("All 5 shared test queries' expected evidence IDs exist in the knowledge base.\n")

demo_query = TEST_QUERIES[0]
demo = retrieve(demo_query["query"], model_infos["small"], "fixed", embeddings,
                chunk_lookup, top_k=3)
print("Query:", demo_query["query"])
print("Expected evidence:", demo_query["expected_evidence_ids"])
print(f"embedding_latency={demo['embedding_latency_seconds']:.4f}s "
      f"retrieval_latency={demo['retrieval_latency_seconds']:.4f}s")
for chunk in demo["chunks"]:
    print(" ", chunk["chunk_id"], round(chunk["score"], 4), chunk["text"][:90])


All 5 shared test queries' expected evidence IDs exist in the knowledge base.

Query: I have a big semester paper due in a few weeks. How should I break it down into tasks in CozyDay?
Expected evidence: ['P002', 'P001']
embedding_latency=0.0058s retrieval_latency=0.0110s
  FIXED_002 0.6772 A semester paper can be represented as a sequence of manageable tasks: choose a topic, fin
  FIXED_003 0.6046 Categories help students see different commitments without treating every item as equally 
  FIXED_001 0.5987 A useful task describes one clear outcome rather than a vague intention. For example, repl


## 6. Five shared test queries (Step 2.1)
These five questions cover five different knowledge-base topics and every one has answerable evidence in the dataset (checked above by `validate_test_queries`). The same five queries, in this same wording, are used for every experiment in Stages 2 and 3 -- see `queries.py` for the full `expected_evidence_notes` on why each evidence paragraph answers its query.

In [6]:
for item in TEST_QUERIES:
    print(f"{item['query_id']}: {item['query']}")
    print(f"  topics: {item['topics']}")
    print(f"  expected evidence: {item['expected_evidence_ids']}\n")


Q1: I have a big semester paper due in a few weeks. How should I break it down into tasks in CozyDay?
  topics: ['Task Creation']
  expected evidence: ['P002', 'P001']

Q2: I have three assignments due this week and they all feel urgent. How do I decide what to work on first?
  topics: ['Priorities', 'Common Questions']
  expected evidence: ['P006', 'P005', 'P025']

Q3: How do I build a weekly study plan and prepare for my exam using spaced review?
  topics: ['Study Planning']
  expected evidence: ['P011', 'P012']

Q4: I want to share my group project plan with teammates. How do I do that without exposing private information?
  topics: ['Privacy']
  expected evidence: ['P016', 'P015']

Q5: How should I use my task completion analytics to write a weekly progress report?
  topics: ['Analytics', 'Reports']
  expected evidence: ['P019', 'P020', 'P021']



## 7. Generation pipeline (Step 1.5)
One locked generation model and decoding configuration for every experiment in this assignment: `Qwen/Qwen2.5-3B-Instruct`, previously evaluated as Model 9 in the team's A7 multi-model assignment, with greedy (non-sampling) decoding and a fixed `max_new_tokens`. The prompt is built from a system instruction, the retrieved context, and the user query; the system instruction explicitly tells the model to rely on the context and to acknowledge insufficient evidence rather than guess.

In [7]:
from generation import load_generation_model, describe_generation_model

generation_info = load_generation_model()
print("Generation model loaded:", describe_generation_model(generation_info))


Loading weights:   0%|                                                                                                                                 | 0/434 [00:00<?, ?it/s]

Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 434/434 [00:00<00:00, 8391.31it/s]

Generation model loaded: {'model_name': 'Qwen/Qwen2.5-3B-Instruct', 'device': 'mps', 'load_time_seconds': 7.616279500070959, 'decoding_settings': {'max_new_tokens': 300, 'do_sample': False, 'num_beams': 1}}


## 8. run_rag interface and nine-combination smoke test
`run_rag(query, model_info, strategy, embeddings, chunk_lookup, generation_info, top_k)` ties embedding, retrieval, and generation together behind one interface, returning the retrieved chunks, the final answer, and embedding/retrieval/generation/total latency. The smoke test below runs the same locked generation model and settings, and the first shared query, across all nine (embedding model x chunking strategy) combinations, displaying the retrieved chunks for every combination rather than only the final answer. The full 3x3x5 = 45-row evaluation grid across all five queries is Stage 3's responsibility (Shivani); this smoke test only confirms every combination of this stage's own pipeline actually runs.

In [8]:
from rag import run_smoke_test

smoke_query = TEST_QUERIES[0]
smoke_results = run_smoke_test(smoke_query["query"], model_infos, chunks_by_strategy,
                                embeddings, chunk_lookup, generation_info)
assert len(smoke_results) == 9

for r in smoke_results:
    print(f"=== {r['model_key']} / {r['strategy']} ===")
    print("Retrieved chunks:")
    for c in r["retrieved_chunks"]:
        print("   ", c["chunk_id"], round(c["score"], 4), c["text"][:90])
    print("Final answer:", r["final_answer"])
    print(f"embed={r['embedding_latency_seconds']:.3f}s "
          f"retrieve={r['retrieval_latency_seconds']:.3f}s "
          f"generate={r['generation_latency_seconds']:.3f}s "
          f"total={r['total_latency_seconds']:.3f}s\n")

print("Stage 2 smoke test: all 9 embedding/chunking combinations completed, "
      "each with its own displayed retrieved chunks and final answer.")


=== small / fixed ===
Retrieved chunks:
    FIXED_002 0.6772 A semester paper can be represented as a sequence of manageable tasks: choose a topic, fin
    FIXED_003 0.6046 Categories help students see different commitments without treating every item as equally 
    FIXED_001 0.5987 A useful task describes one clear outcome rather than a vague intention. For example, repl
Final answer: To break down your big semester paper into manageable tasks in CozyDay, you should start by identifying key milestones and actions that contribute to completing the paper. Here's how you can structure these tasks:

1. **Choose a Topic**: Decide on the subject of your paper. This is a milestone indicating you've chosen a direction.
2. **Find Readings**: Collect relevant books, articles, and other resources. This step helps build the foundation of your research.
3. **Draft an Outline**: Create an outline to organize your thoughts and structure your paper. This outlines the main points you will cover.
4. *

## 9. Stage 2 handoff to Stage 3
Frozen Stage 2 interface for Shivani's experiments:

```python
from embedding import load_all_embedding_models, embed_all_combinations, embed_query
from retrieval import build_chunk_lookup, retrieve
from generation import load_generation_model, generate_answer, GENERATION_MODEL, DECODING_SETTINGS
from queries import TEST_QUERIES
from rag import run_rag, run_smoke_test

model_infos = load_all_embedding_models()
embeddings = embed_all_combinations(model_infos, chunks_by_strategy)  # loads from cache if present
chunk_lookup = build_chunk_lookup(chunks_by_strategy)
generation_info = load_generation_model()

# One row of the required 3 x 3 x 5 = 45-row grid:
result = run_rag(TEST_QUERIES[0]["query"], model_infos["small"], "fixed",
                  embeddings, chunk_lookup, generation_info, top_k=3)
```

`run_rag` never varies `GENERATION_MODEL` or `DECODING_SETTINGS` between calls -- Stage 3 should not either. Embeddings cache under `results/embeddings/` keyed by model name and chunk IDs, so changing `chunking.py`'s chunk contents (not expected) would invalidate the cache automatically; changing nothing means Stage 3 reuses the same cached vectors Stage 2 already verified.

# CozyDay P1 A8 -- Stage 3
**Owner: Shivani Shivani** | Part 2 System Experiments, Part 3 Failure Analysis and Improvement

Stage 3 runs the required **3 embedding models x 3 chunking strategies x 5 shared queries = 45 evaluations** with the unchanged Stage 2 pipeline (same generation model `Qwen/Qwen2.5-3B-Instruct`, same greedy decoding, same Top-K = 3, same five queries), then compares embedding sizes, chunking strategies and dataset size, analyses real failures, and tests an improvement.

The experiments themselves are run from the command line because they download and run three embedding models and a 3B generation model (see `README.md`):

```
python experiments.py main                  # 45 recorded evaluations
python experiments.py scaling-retrieval     # retrieval on a 10-paragraph subset
python experiments.py scaling-generation --model large --strategy hybrid
python experiments.py improve-prompt        # improvement variant 1
python experiments.py improve-prompt --variant complete-within-limit   # variant 2
python experiments.py summary               # summary tables
```

Every row is appended to `results/stage3_evaluations.csv` as it finishes. The cells below load that file, so this part of the notebook runs in seconds and needs no model download. Nothing below changes a recorded result.

In [9]:
import json
import pandas as pd
from experiments import COLUMNS, EVAL_CSV, SCALING_SUBSET_IDS
from queries import TEST_QUERIES
from scoring import KEY_POINTS

s3 = pd.read_csv(EVAL_CSV)
assert list(s3.columns) == COLUMNS, "unexpected column layout"
for col in ["context_score", "answer_score", "evidence_recall", "dimensions",
            "embedding_latency_s", "retrieval_latency_s", "generation_latency_s", "total_latency_s"]:
    s3[col] = pd.to_numeric(s3[col], errors="coerce")
main = s3[s3["stage"] == "main"].copy()

print("Recorded rows by stage / variant:")
print(s3.groupby(["stage", "dataset", "variant"]).size().to_string())

# Completion gate: 45 unique configuration-query rows, all with retrieved chunk text.
assert len(main) == 45 and main["run_id"].nunique() == 45
assert set(zip(main.embedding_key, main.chunking, main.query_id)) == {
    (k, s, q["query_id"]) for k in ["small", "medium", "large"]
    for s in ["fixed", "overlap", "hybrid"] for q in TEST_QUERIES}
for texts in main["retrieved_chunk_texts"]:
    chunks = json.loads(texts)
    assert len(chunks) == 3 and all(chunk.strip() for chunk in chunks)
assert (main["top_k"] == 3).all() and main["final_answer"].str.strip().ne("").all()
print("\n45 main rows: 3 models x 3 strategies x 5 queries, each with 3 full retrieved chunks and an answer.")

Recorded rows by stage / variant:
stage        dataset   variant              
improvement  full      complete-within-limit    45
                       concise-complete         45
main         full      baseline                 45
scaling      subset10  baseline                 45
scaling-gen  subset10  baseline                  5

45 main rows: 3 models x 3 strategies x 5 queries, each with 3 full retrieved chunks and an answer.


## 10. Scoring rubric (one rubric for every row)
Scores come from `scoring.py`, are deterministic, and are applied identically to the main grid, the scaling runs and the improvement runs.

* **Context-quality score (0-3):** 3 = every expected evidence paragraph is in the retrieved context; 2 = the primary evidence paragraph is present but not all; 1 = only a secondary one; 0 = none. Evidence IDs are the ones documented in `queries.py`.
* **Answer-quality score (0-3):** the number of the query's three key points that the answer states. Each key point is a short list of phrases taken from the expected evidence paragraphs.
* Also recorded: evidence recall, whether the primary evidence was retrieved, whether the answer says the context is insufficient, similarity scores, four latencies, and automatic notes.

Limits of the rubric: phrase matching rewards content that uses the evidence's wording, so it can miss a correct paraphrase and it favours longer answers that touch more phrases. Cases where this matters are called out in the failure analysis below.

In [10]:
for query in TEST_QUERIES:
    print(f"{query['query_id']}: {query['query']}")
    print("   expected evidence:", query["expected_evidence_ids"])
    for label, phrases in KEY_POINTS[query["query_id"]]:
        print(f"   key point: {label}")

Q1: I have a big semester paper due in a few weeks. How should I break it down into tasks in CozyDay?
   expected evidence: ['P002', 'P001']
   key point: break the paper into concrete steps
   key point: separate milestones from individual actions / observable completion
   key point: estimate durations conservatively, allow extra time, note dependencies
Q2: I have three assignments due this week and they all feel urgent. How do I decide what to work on first?
   expected evidence: ['P006', 'P005', 'P025']
   key point: start from the earliest firm deadline
   key point: estimate remaining work and break the biggest obligation into small actions
   key point: weigh consequences / importance, or renegotiate a flexible deadline
Q3: How do I build a weekly study plan and prepare for my exam using spaced review?
   expected evidence: ['P011', 'P012']
   key point: start from fixed obligations and realistic available time
   key point: spread review across several sessions with retrieval p

## 11. All 45 evaluations
One row per configuration and query. The full retrieved chunk text, similarity scores, final answer, key points stated and notes for every row are in `results/stage3_evaluations.csv`; one complete row is shown after the table.

In [11]:
view = main[["embedding_key", "dimensions", "chunking", "query_id", "retrieved_source_ids",
             "context_score", "answer_score", "generation_latency_s"]].copy()
view["generation_latency_s"] = view["generation_latency_s"].round(1)
print(view.to_string(index=False))

embedding_key  dimensions chunking query_id          retrieved_source_ids  context_score  answer_score  generation_latency_s
        small         384    fixed       Q1           P002;P003;P004;P001              3           2.0                  23.3
        small         384    fixed       Q2 P005;P006;P003;P004;P025;P026              3           2.0                   8.1
        small         384    fixed       Q3           P011;P012;P027;P028              3           2.0                   9.3
        small         384    fixed       Q4                P016;P015;P022              3           3.0                  15.2
        small         384    fixed       Q5           P019;P020;P021;P022              3           2.0                  13.9
        small         384  overlap       Q1           P002;P003;P001;P004              3           3.0                  21.1
        small         384  overlap       Q2      P003;P004;P005;P006;P002              2           2.0                  20.0


In [12]:
row = main[(main.embedding_key == "large") & (main.chunking == "hybrid") & (main.query_id == "Q2")].iloc[0]
print("run_id:", row.run_id)
print("query:", row["query"])
print("retrieved chunk IDs:", row.retrieved_chunk_ids, "| scores:", row.similarity_scores)
for chunk_id, text in zip(row.retrieved_chunk_ids.split(";"), json.loads(row.retrieved_chunk_texts)):
    print(f"\n[{chunk_id}] {text}")
print("\nfinal answer:\n" + row.final_answer)
print("\ncontext_score:", row.context_score, "| answer_score:", row.answer_score,
      "| key points stated:", json.loads(row.key_points_stated))
print(f"latency: embed={row.embedding_latency_s}s retrieve={row.retrieval_latency_s}s "
      f"generate={row.generation_latency_s}s total={row.total_latency_s}s")

run_id: main|full|baseline|large|hybrid|Q2|k3
query: I have three assignments due this week and they all feel urgent. How do I decide what to work on first?
retrieved chunk IDs: HYBRID_003;HYBRID_013;HYBRID_002 | scores: [0.707764, 0.707684, 0.681727]

[HYBRID_003] Priority is a planning aid, not a judgment about personal worth. A task can be urgent because its deadline is close, important because it advances a long-term goal, both, or neither. Consider consequences, deadlines, and effort before labeling work as high priority. A quiz due tomorrow may need attention before an optional project idea, even when the project feels more exciting. Reserve the highest priority for tasks where delay has a meaningful cost. Review priority labels regularly, especially after completing time-sensitive work. Assigning every task the highest priority makes the label useless and can increase stress rather than focus.

When several deadlines compete, identify the earliest firm deadline and estimate how 

## 12. Embedding size (small 384 vs medium 768 vs large 1024 dimensions)
Each mean below averages 15 rows (3 chunking strategies x 5 queries). Retrieval quality is near the ceiling for every model, so differences are small and should not be over-read: one row is 0.07 of a mean here.

In [13]:
by_embedding = main.groupby(["embedding_key", "dimensions"]).agg(
    mean_context=("context_score", "mean"), mean_recall=("evidence_recall", "mean"),
    mean_answer=("answer_score", "mean"),
    median_query_embed_s=("embedding_latency_s", "median"),
    median_retrieval_s=("retrieval_latency_s", "median"),
    median_generation_s=("generation_latency_s", "median")).round(4)
print(by_embedding.to_string())
print("\nMean context score, embedding model x chunking:")
print(main.pivot_table(index="embedding_key", columns="chunking", values="context_score").round(2).to_string())
print("\nMean answer score, embedding model x chunking:")
print(main.pivot_table(index="embedding_key", columns="chunking", values="answer_score").round(2).to_string())

                          mean_context  mean_recall  mean_answer  median_query_embed_s  median_retrieval_s  median_generation_s
embedding_key dimensions                                                                                                       
large         1024              3.0000       1.0000       2.5333                0.1080              0.0004              15.0769
medium        768               3.0000       1.0000       2.4667                0.0818              0.0004              16.0375
small         384               2.8667       0.9556       2.4000                0.1676              0.0004              13.9363

Mean context score, embedding model x chunking:
chunking       fixed  hybrid  overlap
embedding_key                        
large            3.0     3.0      3.0
medium           3.0     3.0      3.0
small            3.0     2.8      2.8

Mean answer score, embedding model x chunking:
chunking       fixed  hybrid  overlap
embedding_key                      

**Answer:** a larger vector did not always perform better. Medium (768) retrieved exactly as well as large (1024): both put every expected evidence paragraph in the top 3 for all 15 rows. Small (384) made the only retrieval misses (Q2 with overlap and hybrid chunks). Answer scores rise slightly with size (2.40, 2.47, 2.53) but the gap is one row in fifteen. Query-embedding latency (about 0.1-0.2 s) and search latency (under 1 ms) are negligible next to generation (about 14-16 s), so size mainly matters for index time and storage, which the system-design stage covers.

## 13. Chunking strategy (fixed vs overlap vs hybrid)

In [14]:
main["context_words"] = main["retrieved_chunk_texts"].map(lambda s: sum(len(t.split()) for t in json.loads(s)))
by_chunking = main.groupby("chunking").agg(
    mean_context=("context_score", "mean"), mean_recall=("evidence_recall", "mean"),
    mean_answer=("answer_score", "mean"), mean_context_words=("context_words", "mean"),
    median_generation_s=("generation_latency_s", "median")).round(3)
print(by_chunking.to_string())
print("\nMean answer score by query and chunking:")
print(main.pivot_table(index="query_id", columns="chunking", values="answer_score").round(2).to_string())
print("\ncorrelation of retrieved-context words with generation latency:",
      round(main["context_words"].corr(main["generation_latency_s"]), 2))

          mean_context  mean_recall  mean_answer  mean_context_words  median_generation_s
chunking                                                                                 
fixed            3.000        1.000        2.400             325.533               15.077
hybrid           2.933        0.978        2.733             624.667               15.932
overlap          2.933        0.978        2.267             619.733                8.123

Mean answer score by query and chunking:
chunking  fixed  hybrid  overlap
query_id                        
Q1         2.33    3.00     3.00
Q2         2.00    3.00     2.33
Q3         2.00    2.00     2.00
Q4         3.00    3.00     2.00
Q5         2.67    2.67     2.00

correlation of retrieved-context words with generation latency: -0.08


**Answer:** hybrid chunking gave the best answers (2.73 vs 2.40 fixed vs 2.27 overlap) even though fixed chunking had the highest retrieval score (3.00). Hybrid returns whole same-topic paragraph groups, so the model sees each recommendation complete. Fixed chunks are small, about 326 words of context per query against about 625 for hybrid, so the three retrieved chunks sometimes leave out a sentence the answer needs. Overlap supplies about as much context as hybrid (about 620 words) but as overlapping windows, and it scored lower (2.27); in the small-model Q2 case an overlap window of off-topic neighbours (P003 and P004) outranked the right paragraphs. With five queries this is an observation, not a proven cause. Generation time is almost unrelated to context length (correlation -0.08); it follows how many tokens the model writes.

## 14. Dataset scaling (10 paragraphs vs the full 30)
The subset keeps ten paragraphs that hold the evidence for the queries (`SCALING_SUBSET_IDS`), so the test measures how many distractors compete, not whether the answer exists. Queries, models, Top-K and scoring are unchanged. The subset omits P005 and P025, which are expected evidence for Q2, so Q2 can never reach context score 3 there; the raw score drop below is therefore an artifact, and the "attainable" table scores only the evidence that exists in the subset.

In [15]:
print("Subset paragraphs:", SCALING_SUBSET_IDS)
raw = pd.read_csv("results/stage3_summary_scaling.csv")
print("\nRaw context/recall, full (30) vs subset (10):")
print(raw.round(3).to_string(index=False))
attainable = pd.read_csv("results/stage3_summary_scaling_attainable.csv")
print("\nRecall of the evidence that exists in the subset (fair comparison):")
print(attainable.round(3).to_string(index=False))

gen = s3[s3["stage"] == "scaling-gen"]
full_same = main[(main.embedding_key == "large") & (main.chunking == "hybrid")]
print("\nGeneration check, large + hybrid, 5 queries:")
print(f"  full 30 paragraphs : answer {full_same.answer_score.mean():.2f}, "
      f"generation {full_same.generation_latency_s.mean():.1f}s")
print(f"  subset 10 paragraphs: answer {gen.answer_score.astype(float).mean():.2f}, "
      f"generation {gen.generation_latency_s.astype(float).mean():.1f}s")

Subset paragraphs: ['P001', 'P002', 'P006', 'P011', 'P012', 'P015', 'P016', 'P019', 'P020', 'P021']

Raw context/recall, full (30) vs subset (10):
embedding_key chunking  full_context  full_recall  subset_context  subset_recall  context_change_full_minus_subset
        large    fixed           3.0        1.000             2.8          0.867                               0.2
        large   hybrid           3.0        1.000             2.8          0.867                               0.2
        large  overlap           3.0        1.000             2.8          0.867                               0.2
       medium    fixed           3.0        1.000             2.8          0.867                               0.2
       medium   hybrid           3.0        1.000             2.8          0.867                               0.2
       medium  overlap           3.0        1.000             2.8          0.867                               0.2
        small    fixed           3.0        1.00

**Answer:** with these five queries, retrieval quality was identical at 10 and 30 paragraphs (attainable recall 1.0 in all nine configurations). The large + hybrid generation check moved from 2.6 to 2.4 on five rows, within normal variation. Thirty paragraphs is too small to show retrieval degrading; no claim is made about larger collections.

## 15. Failure analysis
The baseline pipeline scored 111 of a possible 135 key points. Before choosing individual cases, the cell below checks where all 24 missed key points came from: whether the evidence was in the retrieved context or not.

In [16]:
from scoring import ends_cleanly

records_missed = []
for _, r in main.iterrows():
    stated = set(json.loads(r.key_points_stated))
    context_text = " ".join(json.loads(r.retrieved_chunk_texts)).lower()
    for label, phrases in KEY_POINTS[r.query_id]:
        if label not in stated:
            records_missed.append({"query_id": r.query_id, "missed_key_point": label,
                                   "evidence_in_retrieved_context": any(p in context_text for p in phrases),
                                   "answer_cut_off": not ends_cleanly(r.final_answer)})
missed = pd.DataFrame(records_missed)
print("missed key points:", len(missed))
print(missed.groupby(["query_id", "missed_key_point", "evidence_in_retrieved_context", "answer_cut_off"]).size().to_string())
print("\nanswers cut off mid-sentence at the locked 300-token limit:",
      int((~main.final_answer.map(ends_cleanly)).sum()), "of 45; by query:",
      main.assign(cut=~main.final_answer.map(ends_cleanly)).groupby("query_id").cut.sum().to_dict())

missed key points: 24
query_id  missed_key_point                                                             evidence_in_retrieved_context  answer_cut_off
Q1        estimate durations conservatively, allow extra time, note dependencies       True                           True              2
Q2        estimate remaining work and break the biggest obligation into small actions  True                           False             4
          weigh consequences / importance, or renegotiate a flexible deadline          True                           False             1
Q3        start from fixed obligations and realistic available time                    True                           False             9
Q4        share only what teammates need (deliverable, person, deadline)               True                           False             3
Q5        completion counts are descriptive, not a measure of effort or success        True                           False             3
          report 

In [17]:
def show_case(run_id, title, observed, cause):
    row = main[main.run_id == run_id].iloc[0]
    query = next(q for q in TEST_QUERIES if q["query_id"] == row.query_id)
    print("=" * 100)
    print(title, "|", run_id)
    print("Original query:", row["query"])
    print("Expected evidence:", query["expected_evidence_ids"], "-", query["expected_evidence_notes"])
    print("Retrieved:", row.retrieved_chunk_ids, "(source paragraphs", row.retrieved_source_ids + ")",
          "scores", row.similarity_scores)
    for chunk_id, text in zip(row.retrieved_chunk_ids.split(";"), json.loads(row.retrieved_chunk_texts)):
        print(f"  [{chunk_id}] {text}")
    print("Answer:\n" + row.final_answer)
    print(f"Scores: context {row.context_score}, answer {row.answer_score}; key points stated: "
          f"{json.loads(row.key_points_stated)}")
    print("OBSERVED FAILURE:", observed)
    print("ROOT CAUSE:", cause)
    print()

show_case("main|full|baseline|large|fixed|Q3|k3", "Failure 1 (Q3)",
          "The answer never says to start from fixed obligations and available time, although that sentence was in the first retrieved chunk. It also borrows a sentence about conservative estimates and unpredictable research from the third chunk, a Task Creation paragraph (P002) that has nothing to do with exam study.",
          "Generation-model limitation and prompt design, not retrieval. Evidence: the first retrieved chunk contains 'fixed obligations such as class meetings, work shifts, and appointments', and this omission happens in all 9 configurations of Q3 (every embedding model and chunking strategy). The prompt does not tell the model to ignore irrelevant passages, so the P002 chunk leaks into the answer.")

Failure 1 (Q3) | main|full|baseline|large|fixed|Q3|k3
Original query: How do I build a weekly study plan and prepare for my exam using spaced review?
Expected evidence: ['P011', 'P012'] - P011 covers building a realistic weekly study plan around fixed obligations; P012 covers spaced, retrieval-practice-based exam preparation across sessions.
Retrieved: FIXED_011;FIXED_012;FIXED_002 (source paragraphs P011;P012;P002) scores [0.743296, 0.740356, 0.625667]
  [FIXED_011] A weekly plan begins with fixed obligations such as class meetings, work shifts, and appointments. Estimate the time left for reading, assignments, practice, and recovery rather than assuming every empty hour is available. Place demanding work in periods when attention is generally strongest, and divide long sessions with breaks. A study block should specify an achievable action such as reviewing lecture notes or completing ten practice problems. Leave unscheduled capacity for interruptions and tasks that take longer than 

In [18]:
show_case("main|full|baseline|large|fixed|Q1|k3", "Failure 2 (Q1)",
          "The answer is cut off in the middle of a sentence. Q1 answers are cut off in all 9 configurations, and the dropped tail loses the 'estimate durations conservatively' key point in two of them.",
          "Prompt design combined with the locked decoding limit. Retrieval is perfect for Q1 (context score 3 in all 9 configurations). The baseline prompt invites long numbered explanations (Q1 answers average 232 words), which cannot finish inside the locked max_new_tokens of 300.")

Failure 2 (Q1) | main|full|baseline|large|fixed|Q1|k3
Original query: I have a big semester paper due in a few weeks. How should I break it down into tasks in CozyDay?
Expected evidence: ['P002', 'P001'] - P002 describes splitting a semester paper into milestone/action steps with dependencies and conservative duration estimates; P001 gives the general rule that large assignments should become several smaller, checkable tasks.
Retrieved: FIXED_001;FIXED_002;FIXED_003 (source paragraphs P001;P002;P003;P004) scores [0.768296, 0.746544, 0.677153]
  [FIXED_001] A useful task describes one clear outcome rather than a vague intention. For example, replace “work on biology” with “summarize chapter four and answer the practice questions.” When preparing a task in CozyDay, give it a short descriptive title, optional notes, an estimated duration, and a realistic due date if one exists. Record the next physical action in the notes when the work feels overwhelming. Large assignments should become s

In [19]:
show_case("main|full|baseline|small|overlap|Q2|k3", "Failure 3 (Q2)",
          "Retrieval found the Priorities evidence (P005, P006) but missed the 'What should I work on first today?' paragraph (P025); the top-ranked chunk was a Categories window (P003 and P004) and the third was a Task Creation window (P002 and P003). The answer then invents an example about a course grade and a scholarship that is nowhere in the context, and says an assignment with work already done is less urgent, which is not what the evidence says.",
          "Mostly embedding choice and chunking, then generation. Evidence: with the small model the overlap and hybrid configurations score context 2 on Q2, while the medium and large models retrieve P005, P006 and P025 for the same query (see the Q2 rows in section 11). In the small model's ranking the Categories window (P003 and P004) scores 0.555, above the Priorities window (P005 and P006) at 0.538, and P025 never reaches the top 3, so the 384-dimension vectors do not separate these topics well. The small model with fixed chunks did retrieve P025. The invented example is a generation limitation: the instruction to use only the context did not stop it.")

Failure 3 (Q2) | main|full|baseline|small|overlap|Q2|k3
Original query: I have three assignments due this week and they all feel urgent. How do I decide what to work on first?
Expected evidence: ['P006', 'P005', 'P025'] - P006 gives the concrete procedure for competing deadlines (earliest firm deadline, remaining work, renegotiate if possible); P005 explains urgency vs. importance; P025 answers the closely related 'what first today' question.
Retrieved: OVERLAP_003;OVERLAP_004;OVERLAP_002 (source paragraphs P003;P004;P005;P006;P002) scores [0.554784, 0.538461, 0.533598]
  [OVERLAP_003] Categories help students see different commitments without treating every item as equally urgent. Common CozyDay contexts include coursework, research, work shifts, personal errands, health, and social plans. Assign a category based on the task’s purpose rather than simply where it takes place. A reading assignment for a class belongs in coursework even when completed in a library. Consistent labels make

## 16. Improvement: change only the prompt
The three failures point at the prompt: answers too long to finish, irrelevant chunks leaking in, and invented examples. The generation model, decoding settings, retrieval, Top-K and queries stay identical; only the system instruction changes (`PROMPT_VARIANTS` in `experiments.py`). Each variant reruns all 45 evaluations, scored with the same rubric. Two variants were tried, and both are reported. Choosing between two variants on the same five queries is mild tuning on the evaluation set, so treat the numbers as indicative.

* **Before:** the Stage 2 prompt, unchanged.
* **Variant 1, `concise-complete`:** at most five short numbered steps, under 150 words, ignore irrelevant passages, no content that is not in the context.
* **Variant 2, `complete-within-limit`:** a complete answer of at most 170 words, in the context's own wording, beginning with the first recommended step, with no content that is not in the context.

In [20]:
imp = s3[s3["stage"] == "improvement"]
before = main.assign(version="before: baseline prompt")
after = imp.assign(version="after: " + imp["variant"])
both = pd.concat([before, after])
both["answers_end_cleanly"] = both["final_answer"].map(ends_cleanly).astype(float)
both["answer_words"] = both["final_answer"].str.split().str.len()
summary = both.groupby("version").agg(
    rows=("run_id", "count"), mean_context=("context_score", "mean"),
    mean_answer=("answer_score", "mean"), answers_end_cleanly=("answers_end_cleanly", "mean"),
    mean_answer_words=("answer_words", "mean"), mean_generation_s=("generation_latency_s", "mean")).round(3)
print("Overall, 45 rows per version:")
print(summary.to_string())
print("\nMean answer score by query:")
print(both.pivot_table(index="query_id", columns="version", values="answer_score").round(2).to_string())
print("\nMean answer score by chunking:")
print(both.pivot_table(index="chunking", columns="version", values="answer_score").round(2).to_string())

Overall, 45 rows per version:
                              rows  mean_context  mean_answer  answers_end_cleanly  mean_answer_words  mean_generation_s
version                                                                                                                 
after: complete-within-limit    45         2.956        2.222                1.000            105.956              9.883
after: concise-complete         45         2.956        2.200                1.000             73.200              7.355
before: baseline prompt         45         2.956        2.467                0.711            152.578             13.615

Mean answer score by query:
version   after: complete-within-limit  after: concise-complete  before: baseline prompt
query_id                                                                                
Q1                                2.22                     2.56                     2.78
Q2                                2.00                     1.78      

In [21]:
key = ["embedding_key", "chunking", "query_id"]
base = main.set_index(key)
variants = {name: imp[imp.variant == name].set_index(key) for name in imp.variant.unique()}
truncated = base[~base.final_answer.map(ends_cleanly)].index
print(f"The {len(truncated)} baseline answers that were cut off mid-sentence:")
print("  baseline answer score:", round(base.loc[truncated].answer_score.mean(), 2),
      "| ends cleanly:", base.loc[truncated].final_answer.map(ends_cleanly).mean())
for name, frame in variants.items():
    print(f"  {name}: answer score {frame.loc[truncated].answer_score.mean():.2f} | ends cleanly: "
          f"{frame.loc[truncated].final_answer.map(ends_cleanly).mean()}")

print("\nKey points stated, out of 9 configurations per query:")
rows = []
for query_id, points in KEY_POINTS.items():
    for label, _ in points:
        entry = {"query": query_id, "key point": label[:60]}
        for version, frame in [("before", base)] + list(variants.items()):
            sub = frame.xs(query_id, level="query_id")
            entry[version] = sum(label in json.loads(v) for v in sub.key_points_stated)
        rows.append(entry)
print(pd.DataFrame(rows).to_string(index=False))

print("\nRows better / same / worse than baseline:")
for name, frame in variants.items():
    delta = frame.answer_score.astype(float) - base.answer_score.astype(float)
    print(f"  {name}: better {(delta > 0).sum()}, same {(delta == 0).sum()}, worse {(delta < 0).sum()}")

The 13 baseline answers that were cut off mid-sentence:
  baseline answer score: 2.69 | ends cleanly: 0.0
  concise-complete: answer score 2.23 | ends cleanly: 1.0
  complete-within-limit: answer score 2.31 | ends cleanly: 1.0

Key points stated, out of 9 configurations per query:
query                                                    key point  before  concise-complete  complete-within-limit
   Q1                          break the paper into concrete steps       9                 9                      9
   Q1 separate milestones from individual actions / observable com       9                 5                      7
   Q1 estimate durations conservatively, allow extra time, note de       7                 9                      4
   Q2                        start from the earliest firm deadline       9                 6                      9
   Q2 estimate remaining work and break the biggest obligation int       5                 4                      7
   Q2 weigh consequenc

In [22]:
for run_key, label in [(("small", "overlap", "Q2"), "Failure 3 row (invented scholarship example)"),
                        (("large", "fixed", "Q3"), "Failure 1 row (fixed obligations missing)"),
                        (("large", "fixed", "Q1"), "Failure 2 row (answer cut off)")]:
    print("=" * 100)
    print(label, run_key)
    print(f"\n--- BEFORE (baseline prompt) | answer score {base.loc[run_key].answer_score}")
    print(base.loc[run_key].final_answer)
    for name, frame in variants.items():
        print(f"\n--- AFTER ({name}) | answer score {frame.loc[run_key].answer_score}")
        print(frame.loc[run_key].final_answer)
    print()

Failure 3 row (invented scholarship example) ('small', 'overlap', 'Q2')

--- BEFORE (baseline prompt) | answer score 2.0
To decide what to work on first among three assignments that feel urgent, you should consider their deadlines, importance, and potential impact on your overall goals. Start by identifying the earliest deadline among the three assignments. This will give you a sense of urgency and help you prioritize tasks based on their immediate deadlines.

Next, assess the importance of each assignment. Determine which one is most crucial towards achieving your long-term goals. For instance, if one assignment is for a course that contributes significantly to your grade, while another is for a project that might lead to a scholarship, the latter could be more important despite being less urgent.

Additionally, evaluate the amount of work remaining for each assignment. If one assignment has a significant portion already completed, it might be less urgent compared to an assignment whe

## 17. Conclusions from Stage 3
* **Retrieval is not the bottleneck here.** Context quality is 2.96 out of 3 overall, and every one of the 24 missed key points had its evidence in the retrieved context. The weak spot is generation: omissions, a long-winded style that runs into the token limit, and occasional invented content.
* **Embedding size:** a larger vector was not always better. Medium matched large; small caused the only retrieval misses. Differences in answer score are within one row in fifteen.
* **Chunking:** hybrid chunking gave the best answers, fixed gave the best retrieval score, overlap added duplicated and off-topic context.
* **Scaling:** retrieval was unchanged between 10 and 30 paragraphs; the corpus is too small to show degradation.
* **Improvement:** both prompt variants made every answer finish (71 percent to 100 percent), removed the invented example in the Q2 failure row, and cut mean generation time by 27 to 46 percent. Neither raised the mean answer score (2.47 before; 2.20 and 2.22 after), because shorter answers drop details the rubric looks for, and the 13 answers that used to be cut off scored lower, not higher, once they were shortened. Variant 2 fixed the Q3 omission in 3 of 9 configurations, variant 1 improved the Q4 sharing point from 6 to 9 of 9. So the prompt change fixed completeness and reliability but not coverage, and it is a trade-off, not a clear win. Recovering coverage without losing completeness needs a longer decoding limit, which the contract locks, or a different lever such as re-ranking to drop off-topic chunks before generation; both are left as next steps.